In [32]:
import torch

H = 721
W = 1441

# Step sizes
lat_step = 180.0 / (H - 1)    # 0.25°
lon_step = 360.0 / (W - 1)    # 0.25°

# Latitudes: center of each cell
lat_tensor = torch.linspace(-90.0, 90.0, H, dtype=torch.float32) # - lat_step/2    # the - sign assumes -90 will be removed, because 721 is ugly.
print(lat_tensor[0], lat_tensor[-1], lat_tensor[1]-lat_tensor[0])
# Longitudes: center of each cell
lon_tensor = torch.linspace(-180.0, 180.0, W, dtype=torch.float32) # - lon_step/2    # the - sign assumes -180 will be removed, -180 and 180 are the same.
print(lon_tensor[0], lon_tensor[-1], lon_tensor[1]-lon_tensor[0])

lat_tensor = lat_tensor[:]
lon_tensor = lon_tensor[:-1]   # the +180 is removed

# Broadcast to grids
lat_grid = lat_tensor.unsqueeze(1).expand(H, W-1)
lon_grid = lon_tensor.unsqueeze(0).expand(H, W-1)

# Stack into (2, H, W)
lat_lon_tensor = torch.stack([lat_grid, lon_grid], dim=0)   # (2, 721, 1440), with a resolution of 0.25 degree

print(lat_lon_tensor.shape)
print(lat_lon_tensor[0, 360, :])
print(lat_lon_tensor[1, :, 0])

tensor(-90.) tensor(90.) tensor(0.2500)
tensor(-180.) tensor(180.) tensor(0.2500)
torch.Size([2, 721, 1440])
tensor([0., 0., 0.,  ..., 0., 0., 0.])
tensor([-180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -180.,
        -180., -180., -180., -180., -180., -180., -180., -180., -180., -

In [1]:
import os
import zipfile
import tempfile
import numpy as np
import requests
import geopandas as gpd
import rasterio
from rasterio import features
import xarray as xr

def download_and_extract_naturalearth_land(dest_dir):
    """
    Download the Natural Earth 10m land polygon shapefile and unzip it.
    """
    # URL for 10m land shapefile (Natural Earth)
    url = "https://naciscdn.org/naturalearth/10m/physical/ne_10m_land.zip"  # from Natural Earth site :contentReference[oaicite:0]{index=0}
    zip_path = os.path.join(dest_dir, "ne_10m_land.zip")

    # Download if needed
    if not os.path.exists(zip_path):
        print(f"Downloading {url} …")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")

    # Extract
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)
    print("Extracted shapefile.")

    # Find the .shp path
    shp_path = None
    for fname in os.listdir(dest_dir):
        if fname.endswith(".shp") and "land" in fname:
            shp_path = os.path.join(dest_dir, fname)
            break
    if shp_path is None:
        raise RuntimeError("Could not find .shp in the extracted zip.")
    print("Shapefile:", shp_path)
    return shp_path

def create_land_sea_mask_era5(H=721, W=1440, all_touched=False):
    """
    Create a land-sea mask (1 = land, 0 = sea) on an ERA5-style grid.
    """
    # Create a temporary folder to download shapefile
    with tempfile.TemporaryDirectory() as tmpdir:
        shp = download_and_extract_naturalearth_land(tmpdir)
        land_gdf = gpd.read_file(shp)
        land_gdf = land_gdf.to_crs("EPSG:4326")  # ensure WGS84

        # Create geometry list for rasterization
        geoms = ((geom, 1) for geom in land_gdf.geometry)

        # Build transform for ERA5 grid
        transform = rasterio.transform.from_bounds(
            west=-180, south=-90, east=180, north=90, width=W, height=H
        )

        mask = features.rasterize(
            geoms,
            out_shape=(H, W),
            transform=transform,
            fill=0,
            all_touched=all_touched,
            dtype=np.uint8,
        )

    return mask

def save_mask(mask, lat, lon, npy_path=None, nc_path=None):
    """
    Save mask as .npy or NetCDF.
    lat, lon: 1D arrays of lat/lon coordinate centers
    """
    if npy_path:
        np.save(npy_path, mask)
        print("Saved npy to", npy_path)
    if nc_path:
        ds = xr.Dataset(
            {"lsm": (("lat", "lon"), mask)},
            coords={"lat": lat, "lon": lon},
        )
        ds.to_netcdf(nc_path)
        print("Saved NetCDF to", nc_path)

H, W = 721, 1440
lat = np.linspace(-90, 90, H)
lon = np.linspace(-180, 180, W)

mask = create_land_sea_mask_era5(H=H, W=W, all_touched=False)
# Flip to match lat = -90..90 (row 0 = south)
mask = np.flipud(mask)
save_mask(mask, lat, lon, npy_path="land_sea_mask_721x1440.npy", nc_path="land_sea_mask_721x1440.nc")

print("Land count:", int((mask == 1).sum()), "Sea count:", int((mask == 0).sum()))


Download complete.
Extracted shapefile.
Shapefile: C:\Users\Mike\AppData\Local\Temp\tmpa510ha5c\ne_10m_land.shp
Saved npy to land_sea_mask_721x1440.npy
Saved NetCDF to land_sea_mask_721x1440.nc
Land count: 343316 Sea count: 694924


In [ ]:
import torch
import math

def Zeus_Pair_Position(H, W, i, j):
    """
    Compute absolute position embedding for a given lat-lon grid cell (i,j).
    
    Args:
        H (int): number of latitude points (height)
        W (int): number of longitude points (width)
        i (int): row index (0 <= i < H)
        j (int): column index (0 <= j < W)
        
    Returns:
        torch.Tensor: shape [5], [cos(lat), sin(lat), cos(lon), sin(lon), cos(lat2)]
    """
    # Latitude: 90 to -90
    lat = 90 - 180 * i / (H - 1)
    lon = -180 + 360 * j / (W - 1)
    
    lat_rad = math.radians(lat)
    lon_rad = math.radians(lon)
    
    # Example: lat2 could be some shifted latitude (just demonstration)
    lat2_rad = lat_rad  # you can modify if needed
    
    emb = torch.tensor([
        math.cos(lat_rad),
        math.sin(lat_rad),
        math.cos(lon_rad),
        math.sin(lon_rad),
    ], dtype=torch.float32)
    
    return emb

# Example usage
H, W = 180, 360
i, j = 0, 180
pos_emb = Zeus_Pair_Position(H, W, i, j)
print(pos_emb)

tensor([6.1232e-17, 1.0000e+00, 9.9996e-01, 8.7508e-03, 6.1232e-17])


In [63]:
import pvlib
import pandas as pd

lat, lon = 40.0, -105.0
time = pd.Timestamp("2024-06-21 12:00:00", tz="UTC")

solpos = pvlib.solarposition.get_solarposition(time, lat, lon)
zenith_angle = solpos['zenith']
print(zenith_angle)


from datetime import datetime
solpos = pvlib.solarposition.get_solarposition(pd.Timestamp(datetime(2024, 6, 21, 12), tz="UTC"), lat, lon)
zenith_angle = solpos['zenith'].item()
print(zenith_angle)


2024-06-21 12:00:00+00:00    86.09953
Name: zenith, dtype: float64
86.0995302043978


In [52]:
class Snapshot:
    def __init__(self, device='cpu'):
        self._device = device
        self._fresh = True
    def _get_device(self):
        return self._device
    def _set_device(self, device):
        self._device = device
    device = property(
        fget = _get_device,
        fset = _set_device,
    )
    def _get_fresh(self):
        return self._fresh
    def _set_fresh(self, fresh):
        self._fresh = fresh
    fresh = property(
        fget = _get_fresh,
        fset = _set_fresh,
    )    
    def to(self, device):
        self._device = device
    def delete(self):
        self._device = 'cpu'

from copy import copy
def take_substep(md, X0, X1, Y, cascade_step):
    _Y = copy(Y); _Y.fresh = False
    return _Y, 1.0, [2.0, 3.0]

nCascades = 32
snapshots = [Snapshot('cpu') for _ in range(nCascades+2)]
device = "cuda"
md = {}

#------- Mike1 algorithm, a 6-hourly training algorithm that mitigates the bottleneck on data pipeline ------
# It takes 528 training substeps, with varing cascade steps, with 2 + 32 raw data X.
# The distribution of cascase steps is: 
# [32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
# = [1-cascade's training steps, 2- ..., ... 32-cascade's trainig steps]
# 528 * 0.2 sec == 106 sec. The data pipeline should load 25GB (read below) of data withinn 106 sec, 250MB/s.
# 250MB/s in 4 drives: 60MB/s
# The GPU thread(s) will no longer wait for data threads to finish loading a batch of training data, saving GPU cost.
# We might not need an 4x expensive NVEm drive.
# 32 * 6 = 8 days * 24 hours, it trains to forecast 8 future days
# The maximum number of data X's that are simultaneously on cuda is 66,
# At most 66 * 4MB * 95 variables = 25GB of GPU memory is taken by data
# The data pipeline may simultaneously have at most 2 * (2 + 32) = 68 raw X's, taking 25GB of cpu-side memory.
# Quite feasible on a 100GB-GPU virtual machine.
# 528 training substeps with 34 raw X's, ... might allow multi-GPU training, although GPU-loading of data may conflict a lot.

assert len(snapshots) >= 2, "length of Xs should be >= 2"
X0, X1 = snapshots[0], snapshots[1]; X0.to(device); X1.to(device); snapshots = snapshots[2:]
oldQ = [(X0, X1)]

assert len(snapshots) > 0, "length of Xs should be >= 3"

nSubsteps = 0
nSubsteps_by_cascade = [0] * nCascades
step_loss = []; step_metrics = []
while len(snapshots) > 0:  # loops at least once
    # State: 
    # len(Xs) > 0, len(oldQ) > 0, [x0.device == device and x1.device == device for (x0, x1) in oldQ]
    # 2 * len(oldQ) count of X is stored in oldQ in device, together with model parameters.
    # oldQ[0] stores raw X's
        
    Y = snapshots[0]; Y.to(device); snapshots = snapshots[1:]
    newQ = [(oldQ[0][1], Y)]  # assert len(newQ) > 0        # newQ[0] stores raw X's

    #-------------------- State check, no-remove ------------------
    nXOnDevice = 0
    assert len([x for x in snapshots if x.device == device]) == 0, "Some of Xs are on device"
    assert len(oldQ) > 0, "oldQ is empty"
    check = [x0.device == device and x1.device == device for (x0, x1) in oldQ]
    assert all(check), "Some pairs in oldQ are not on device"
    nXOnDevice += 2 * len(oldQ)
    assert len(newQ) == 1
    check = [x0.device == device and x1.device == device for (x0, x1) in newQ]
    assert all(check), "The only pair in newQ are not on device"
    nXOnDevice += 2
    print(f"n of X that are on device: {nXOnDevice}, accum substeps taken: {nSubsteps}")

    substep_loss = []; substep_metrics = []
    while len(oldQ) > 0:     # loops at least once
        X0, X1 = oldQ[0]; oldQ = oldQ[1:]    # assert X.device == device
        cascade_step = len(newQ)
        step_result, loss, metrics = take_substep(md, X0, X1, Y, cascade_step) # forward, backward, and optimizer.step
        nSubsteps += 1
        nSubsteps_by_cascade[cascade_step-1] += 1
        assert step_result.device == device

        newQ.append((X1, step_result))                      # +1 X
        X0.delete()  # del all and clean cuda immediately   # -1 X
        
        substep_loss.append(loss)
        substep_metrics.append(metrics)
        
    step_loss.append(substep_loss)
    step_metrics.append(substep_metrics)

    # state: len(oldQ) == 0, len(oldQ) > 0
    oldQ = newQ
print("nSubsteps", nSubsteps)
print("nSubsteps_by_cascade", nSubsteps_by_cascade)
assert nSubsteps == int((nCascades+1)*(nCascades)/2)


n of X that are on device: 4, accum substeps taken: 0
n of X that are on device: 6, accum substeps taken: 1
n of X that are on device: 8, accum substeps taken: 3
n of X that are on device: 10, accum substeps taken: 6
n of X that are on device: 12, accum substeps taken: 10
n of X that are on device: 14, accum substeps taken: 15
n of X that are on device: 16, accum substeps taken: 21
n of X that are on device: 18, accum substeps taken: 28
n of X that are on device: 20, accum substeps taken: 36
n of X that are on device: 22, accum substeps taken: 45
n of X that are on device: 24, accum substeps taken: 55
n of X that are on device: 26, accum substeps taken: 66
n of X that are on device: 28, accum substeps taken: 78
n of X that are on device: 30, accum substeps taken: 91
n of X that are on device: 32, accum substeps taken: 105
n of X that are on device: 34, accum substeps taken: 120
n of X that are on device: 36, accum substeps taken: 136
n of X that are on device: 38, accum substeps taken:

In [10]:
from skyfield import api, almanac
import datetime

ts = api.load.timescale()
eph = api.load('de421.bsp')

def seasons_for_year(year):
    t0 = ts.utc(year, 1, 1)
    t1 = ts.utc(year, 12, 31)
    times, events = almanac.find_discrete(t0, t1, almanac.seasons(eph))
    # events: 0=March equinox, 1=June solstice, 2=Sept equinox, 3=Dec solstice
    results = {}
    for ti, ev in zip(times, events):
        dt = ti.utc_datetime()
        if ev == 0:
            results['mar_equinox'] = dt
        elif ev == 1:
            results['jun_solstice'] = dt
        elif ev == 2:
            results['sep_equinox'] = dt
        elif ev == 3:
            results['dec_solstice'] = dt
    return results

print(seasons_for_year(2026))


{'mar_equinox': datetime.datetime(2026, 3, 20, 14, 45, 57, 448680, tzinfo=datetime.timezone.utc), 'jun_solstice': datetime.datetime(2026, 6, 21, 8, 24, 30, 347031, tzinfo=datetime.timezone.utc), 'sep_equinox': datetime.datetime(2026, 9, 23, 0, 5, 13, 168825, tzinfo=datetime.timezone.utc), 'dec_solstice': datetime.datetime(2026, 12, 21, 20, 50, 14, 175954, tzinfo=datetime.timezone.utc)}


In [1]:
from datetime import datetime, timedelta

# Define key solar events (UTC)
WINTER_SOLSTICE = datetime(2025, 12, 21, 15, 0)  # Example: 21 Dec 2025 15:00 UTC
SPRING_EQUINOX = datetime(2025, 3, 20, 21, 0)    # 20 Mar 2025 21:00 UTC
SUMMER_SOLSTICE = datetime(2025, 6, 21, 9, 0)    # 21 Jun 2025 09:00 UTC
AUTUMN_EQUINOX = datetime(2025, 9, 23, 6, 0)     # 23 Sep 2025 06:00 UTC

# List in chronological order (starting from winter solstice)
events = [
    WINTER_SOLSTICE,
    SPRING_EQUINOX,
    SUMMER_SOLSTICE,
    AUTUMN_EQUINOX,
    WINTER_SOLSTICE.replace(year=WINTER_SOLSTICE.year + 1)  # to complete the orbit
]

angles = [0, 90, 180, 270, 360]

def orbital_angle(dt: datetime) -> float:
    """
    Map a datetime to an angle 0-360 based on Earth's orbit.
    0° = NH winter solstice, 90° = March equinox, etc.
    """
    # Find which segment dt belongs to
    for i in range(len(events)-1):
        if events[i] <= dt < events[i+1]:
            start, end = events[i], events[i+1]
            angle_start, angle_end = angles[i], angles[i+1]
            fraction = (dt - start) / (end - start)
            angle = angle_start + fraction * (angle_end - angle_start)
            return angle % 360
    # If dt is before first winter solstice
    return orbital_angle(dt.replace(year=dt.year-1))

# Example usage
now = datetime.utcnow()
angle = orbital_angle(now)
print(f"Orbital angle: {angle:.2f}°")

Orbital angle: 281.55°


In [53]:
from skyfield import api, almanac
from datetime import datetime, timezone

# Load ephemeris
ts = api.load.timescale()
eph = api.load('de421.bsp')

def get_season_events(year):
    t0 = ts.utc(year, 1, 1)
    t1 = ts.utc(year, 12, 31)
    times, events = almanac.find_discrete(t0, t1, almanac.seasons(eph))
    season_dict = {}
    for ti, ev in zip(times, events):
        dt = ti.utc_datetime()
        # Ensure UTC-aware
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        if ev == 0:
            season_dict['mar_equinox'] = dt
        elif ev == 1:
            season_dict['jun_solstice'] = dt
        elif ev == 2:
            season_dict['sep_equinox'] = dt
        elif ev == 3:
            season_dict['dec_solstice'] = dt
    return season_dict

def earth_orbit_angle(dt):
    # If naive datetime, treat as UTC
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    
    year = dt.year
    seasons = get_season_events(year)
    
    season_angles = [
        (seasons['dec_solstice'], 0),
        (seasons['mar_equinox'], 90),
        (seasons['jun_solstice'], 180),
        (seasons['sep_equinox'], 270),
        (get_season_events(year + 1)['dec_solstice'], 360)
    ]
    
    for i in range(len(season_angles) - 1):
        t0, a0 = season_angles[i]
        t1, a1 = season_angles[i+1]
        if t0 <= dt < t1:
            fraction = (dt - t0).total_seconds() / (t1 - t0).total_seconds()
            angle = a0 + fraction * (a1 - a0)
            return angle
    return None

# Examples
angle1 = earth_orbit_angle(datetime(2025, 11, 20, 12))  # naive treated as UTC
angle2 = earth_orbit_angle(datetime(2025, 11, 20, 12, tzinfo=timezone.utc))  # aware datetime
print(f"Earth orbital angle 1: {angle1:.2f}°")
print(f"Earth orbital angle 2: {angle2:.2f}°")


Earth orbital angle 1: 281.62°
Earth orbital angle 2: 281.62°


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# Policy network
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.fc(state)

# Environment and setup
env = gym.make("CartPole-v1")
policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
optimizer = optim.Adam(policy.parameters(), lr=1e-2)
gamma = 0.99

for episode in range(500):
    states, actions, rewards = [], [], []
    state = env.reset()[0]
    done = False

    # 1️⃣ Run an episode
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32)
        probs = policy(state_tensor)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        next_state, reward, done, _, _ = env.step(action.item())
        states.append(state_tensor)
        actions.append(action)
        rewards.append(reward)
        state = next_state

    # 2️⃣ Compute returns (discounted rewards)
    returns = []
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # normalize

    # 3️⃣ Compute policy loss
    log_probs = []
    for state, action in zip(states, actions):
        probs = policy(state)
        dist = torch.distributions.Categorical(probs)
        log_probs.append(dist.log_prob(action))
    log_probs = torch.stack(log_probs)

    loss = -(log_probs * returns).sum()

    # 4️⃣ Backpropagate
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Episode {episode}, total reward: {sum(rewards):.2f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

# --- Policy Network ---
class PolicyNet(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(action_dim),
            nn.Softmax(dim=-1)
        )

    def forward(self, state):
        return self.net(state)

# --- REINFORCE Agent ---
def reinforce(env_name="CartPole-v1", lr=1e-3, gamma=0.99, episodes=500):
    env = gym.make(env_name)
    policy = PolicyNet(env.observation_space.shape[0], env.action_space.n)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    for ep in range(episodes):
        log_probs, rewards = [], []
        state = env.reset()[0] if isinstance(env.reset(), tuple) else env.reset()

        done = False
        while not done:
            state_t = torch.tensor(state, dtype=torch.float32)
            probs = policy(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()

            log_probs.append(dist.log_prob(action))

            next_state, reward, done, *_ = env.step(action.item())
            rewards.append(reward)
            state = next_state

        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Compute policy gradient loss
        loss = -torch.sum(torch.stack(log_probs) * returns)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Episode {ep+1}: total reward = {sum(rewards)}")

    env.close()

reinforce()


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

def scrape_interesting_engineering(max_pages=100):
    # Chrome options
    options = Options()
    options.add_argument("--headless")  # comment this line if you want to see the browser
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    # Start driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get("https://interestingengineering.com/news")

    articles = []
    page = 0

    while page < max_pages:
        page += 1
        print(f"📰 Scraping page {page}...")

        # Wait for articles to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "article"))
            )
        except Exception:
            print("❌ No articles found, stopping.")
            break

        # Parse HTML
        soup = BeautifulSoup(driver.page_source, "html.parser")
        found = 0
        for article in soup.select("article"):
            title_tag = article.find("a")
            date_tag = article.find("time")
            if title_tag:
                title = title_tag.get_text(strip=True)
                date = date_tag.get_text(strip=True) if date_tag else "No date"
                articles.append((title, date))
                found += 1
        print(f"  Found {found} articles on this page.")

        # Try to click NEXT
        try:
            next_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//a[contains(translate(., 'next', 'NEXT'), 'NEXT')]"))
            )
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(2)  # wait for next page to load
        except Exception:
            print("✅ Reached last page.")
            break

    driver.quit()
    return articles


def save_to_text(articles, filename="InterestingEngineering_News.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        for title, date in articles:
            f.write(f"{title} — {date}\n")
    print(f"✅ Saved {len(articles)} articles to {filename}")


if __name__ == "__main__":
    data = scrape_interesting_engineering()
    save_to_text(data)


📰 Scraping page 1...
  Found 1 articles on this page.
📰 Scraping page 2...
  Found 1 articles on this page.
📰 Scraping page 3...
  Found 10 articles on this page.
✅ Reached last page.


RemoteDisconnected: Remote end closed connection without response

In [ ]:
You know ResNet and how skipping happens in every block. In ResNet models, skipping happens only for individual blocks. 
I imagine a model where skipping happens for any group of contiguous blocks. 
This will improve smooth specialization between blocks. 
Please code my idea.

In [ ]:
Nice idea — contiguous-group skipping should let neighboring blocks specialize smoothly. 
Below is a ready-to-run PyTorch implementation of a Group-Skip ResNet that implements learned contiguous window gating over successive residual blocks.

In [ ]:
Key idea

A small gating network (an MLP from a global pooled feature) predicts, per sample, a window center c and half-width w.
For block index i we form a differentiable mask
mask_i = sigmoid(k * (w - |i - c|))
so blocks within the window (|i-c| < w) get mask ≈ 1, blocks outside get mask ≈ 0. k controls sharpness.
Each block's output is scaled by its mask before adding into the residual path. When mask ≈ 0 → block is effectively skipped.
At inference you can make masks hard (threshold) if desired.

In [ ]:
The code below provides:
BasicBlock — a simple ResNet-like block
GroupSkipResNet — stacks blocks and applies the group-skip gating
options for temperature k, L1 regularization on width to encourage narrow windows, and hard thresholding at inference

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional

class BasicBlock(nn.Module):
    """A simplified ResNet basic block (no downsample handling here for brevity).
       Assumes input and output channels are equal and stride=1."""
    def __init__(self, channels: int, norm_layer=nn.BatchNorm2d):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = norm_layer(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = norm_layer(channels)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        # standard residual add happens outside (we want to apply mask to out)
        return out


class GroupSkipResNet(nn.Module):
    """ResNet-like backbone with learnable contiguous-group skipping.
       - num_blocks: total number of successive basic blocks
       - channels: channel count kept constant for simplicity
       - k: sharpness of sigmoid window (larger -> closer to hard window)
       - gate_hidden: hidden dim for gating MLP"""
    def __init__(self,
                 num_blocks: int,
                 channels: int = 64,
                 k: float = 8.0,
                 gate_hidden: int = 64,
                 norm_layer=nn.BatchNorm2d):
        super().__init__()
        assert num_blocks >= 1
        self.num_blocks = num_blocks
        self.channels = channels
        self.k = float(k)

        # initial conv / stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=7, stride=2, padding=3, bias=False),
            norm_layer(channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # stack of basic blocks (same channels here for simplicity)
        self.blocks = nn.ModuleList([BasicBlock(channels, norm_layer=norm_layer) for _ in range(num_blocks)])

        # gating MLP: map global pooled features -> center (c) and half-width (w)
        # we predict in a normalized coordinate system where block indices run 0..num_blocks-1
        self.gate_mlp = nn.Sequential(
            nn.Linear(channels, gate_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(gate_hidden, 2)   # outputs: raw_c, raw_w
        )

        # small learnable scale and bias to make initial gating sensible
        # initialize bias such that c ~ (num_blocks-1)/2 and w ~ num_blocks/2 (so all blocks active initially)
        self.register_buffer("block_indices", torch.arange(num_blocks, dtype=torch.float32).unsqueeze(0))  # shape (1, N)
        # We'll use a tiny offset initializer (bias in gate_mlp second layer will handle initial range)

    def forward(self, x, hard_threshold: Optional[float] = None):
        """
        x: input images (B,3,H,W)
        hard_threshold: if provided, threshold masks at this value for hard gating (inference)
        Returns:
          out: tensor after passing through blocks with group-skip
          masks: (B, num_blocks) float masks per sample (soft in training, may be hard if threshold provided)
        """
        B = x.shape[0]
        out = self.stem(x)                 # (B, C, H', W')

        # Global pooled feature for gating (per-sample)
        gp = F.adaptive_avg_pool2d(out, (1, 1)).view(B, -1)  # (B, channels)

        # Predict raw center and width
        raw = self.gate_mlp(gp)           # (B, 2)
        raw_c = raw[:, 0]                 # center
        raw_w = raw[:, 1]                 # half-width (can be negative initially)

        # Map raw outputs to meaningful ranges:
        # center c in [0, num_blocks-1], width w in [0, num_blocks]
        c = (torch.sigmoid(raw_c) * (self.num_blocks - 1)).unsqueeze(1)   # (B,1)
        w = (torch.sigmoid(raw_w) * self.num_blocks).unsqueeze(1)         # (B,1)

        # compute differentiable window masks per block index
        # block_indices shape (1, N), broadcast with (B,1) -> (B,N)
        indices = self.block_indices.to(x.device)  # (1, N)
        dist = torch.abs(indices - c)              # (B, N)
        # mask = sigmoid(k * (w - dist))
        masks = torch.sigmoid(self.k * (w - dist))  # (B, N) in (0,1)

        # Optionally hard-threshold at inference
        if hard_threshold is not None:
            masks = (masks >= float(hard_threshold)).float()

        # sequentially apply blocks with per-sample mask
        # note: mask for block i is shape (B,) -> we reshape to (B,1,1) to broadcast to (B,C,H,W)
        for i, block in enumerate(self.blocks):
            block_out = block(out)  # (B, C, H, W)
            mi = masks[:, i].view(B, 1, 1, 1)  # broadcast
            out = out + block_out * mi

        return out, masks

    def get_mask_regularizer(self, masks):
        """Return a regularizer encouraging compact windows.
           For example, encourage small widths by L1 on masks sum (smaller sum -> narrower).
           masks: (B, N)"""
        # L1 on per-sample mask sums approximates window width
        width_est = masks.sum(dim=1)          # (B,)
        return width_est.mean()               # scalar

# Example usage
if __name__ == "__main__":
    model = GroupSkipResNet(num_blocks=8, channels=32, k=10.0)
    images = torch.randn(4, 3, 224, 224)
    out, masks = model(images)
    print("out shape:", out.shape)         # expected (B, channels, H', W')
    print("masks:", masks)                 # per-sample, per-block masks


out shape: torch.Size([4, 32, 56, 56])
masks: tensor([[0.4084, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9763],
        [0.3933, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9755],
        [0.4038, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9789],
        [0.4029, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.9743]],
       grad_fn=<SigmoidBackward0>)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from s2cnn import S2Convolution, S2Activation

# ------------------------------
# 1️⃣ Spherical CNN Model
# ------------------------------
class SphericalWeatherNet(nn.Module):
    def __init__(self, in_channels=12, out_channels=1, bandwidth=64):
        super().__init__()
        self.conv1 = S2Convolution(in_channels, 32, bandwidth, bandwidth//2)
        self.act1 = S2Activation(nn.ReLU())
        self.conv2 = S2Convolution(32, 64, bandwidth//2, bandwidth//4)
        self.act2 = S2Activation(nn.ReLU())
        self.conv3 = S2Convolution(64, out_channels, bandwidth//4, bandwidth//8)

    def forward(self, x):
        x = self.act1(self.conv1(x))
        x = self.act2(self.conv2(x))
        x = self.conv3(x)
        return x

# ------------------------------
# 2️⃣ Grid + mask helper
# ------------------------------
def make_era5_to_s2cnn_grid_lat_range(lat, lon, bandwidth=64, lat_min=0, lat_max=90):
    b = bandwidth
    beta = np.linspace(0, np.pi, 2*b, endpoint=False)
    alpha = np.linspace(0, 2*np.pi, 2*b, endpoint=False)
    beta_grid, alpha_grid = np.meshgrid(beta, alpha, indexing="ij")
    lat_sph = 90 - np.degrees(beta_grid)
    lon_sph = np.degrees(alpha_grid)
    lon_norm = (lon_sph / 180.0) - 1.0
    lat_norm = -lat_sph / 90.0
    grid = np.stack([lon_norm, lat_norm], axis=-1)
    grid = torch.from_numpy(grid).unsqueeze(0).float().cuda()
    mask = ((lat_sph >= lat_min) & (lat_sph <= lat_max)).astype(np.float32)
    mask = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0).cuda()
    return grid, mask

# ------------------------------
# 3️⃣ ERA5 -> S2CNN conversion
# ------------------------------
def era5_to_s2cnn_batch(X, grid_s2cnn, mask=None):
    is_sequence = (X.dim() == 4)
    if is_sequence:
        T, C, H, W = X.shape
        X_flat = X.reshape(T*C, 1, H, W)
    else:
        C, H, W = X.shape
        X_flat = X.unsqueeze(0)
    X_s2cnn = F.grid_sample(X_flat, grid_s2cnn, align_corners=True)
    if mask is not None:
        X_s2cnn *= mask
    if is_sequence:
        X_s2cnn = X_s2cnn.reshape(T, C, X_s2cnn.shape[-2], X_s2cnn.shape[-1])
    else:
        X_s2cnn = X_s2cnn.squeeze(0)
    return X_s2cnn

# ------------------------------
# 4️⃣ S2CNN -> ERA5 conversion
# ------------------------------
def s2cnn_to_era5_batch(Y_s2cnn, lat, lon):
    is_sequence = (Y_s2cnn.dim() == 5)
    if is_sequence:
        B, T, C, Hs, Ws = Y_s2cnn.shape
        Y_flat = Y_s2cnn.reshape(B*T, C, Hs, Ws)
    else:
        B, C, Hs, Ws = Y_s2cnn.shape
        Y_flat = Y_s2cnn
    H, W = len(lat), len(lon)
    lon_grid, lat_grid = np.meshgrid(lon, lat)
    lon_norm = (lon_grid / 180.0) - 1.0
    lat_norm = -lat_grid / 90.0
    grid = np.stack([lon_norm, lat_norm], axis=-1)
    grid = torch.from_numpy(grid).unsqueeze(0).float().cuda()
    Y_era5 = F.grid_sample(Y_flat, grid, align_corners=True)
    if is_sequence:
        Y_era5 = Y_era5.reshape(B, T, C, H, W)
    return Y_era5

# ------------------------------
# 5️⃣ Dataset (streaming version)
# ------------------------------
class ERA5StreamingDataset(Dataset):
    def __init__(self, file_list, T_in=3, lat_crop=(0,90), bandwidth=64):
        self.file_list = file_list
        self.T_in = T_in
        self.lat_min, self.lat_max = lat_crop
        self.bandwidth = bandwidth

        # Load grid from first file
        with h5py.File(file_list[0], 'r') as f:
            self.lat = f['lat'][:]
            self.lon = f['lon'][:]

        # Precompute S2CNN grid & mask
        self.grid_s2cnn, self.mask = make_era5_to_s2cnn_grid_lat_range(
            self.lat, self.lon, bandwidth=self.bandwidth,
            lat_min=self.lat_min, lat_max=self.lat_max
        )
        self.grid_s2cnn = self.grid_s2cnn.cuda()
        self.mask = self.mask.cuda()

        # Compute sample offsets
        self.sample_offsets = []
        offset = 0
        for fname in file_list:
            with h5py.File(fname,'r') as f:
                N = f['X'].shape[0]
                self.sample_offsets.append((offset, offset+N))
                offset += N
        self.total_samples = offset

    def __len__(self):
        return self.total_samples - self.T_in

    def __getitem__(self, idx):
        for file_idx, (start,end) in enumerate(self.sample_offsets):
            if start <= idx < end - self.T_in:
                idx_in_file = idx - start
                file_name = self.file_list[file_idx]
                break
        else:
            raise IndexError(f"{idx} out of range")

        with h5py.File(file_name,'r') as f:
            X_seq = f['X'][idx_in_file:idx_in_file+self.T_in]
            Y = f['Y'][idx_in_file+self.T_in]

        X_seq = torch.from_numpy(X_seq).float()
        Y = torch.from_numpy(Y).float()
        X_s2cnn = era5_to_s2cnn_batch(X_seq, self.grid_s2cnn, self.mask)
        return X_s2cnn, Y

# ------------------------------
# 6️⃣ Recursive multi-day forecast
# ------------------------------
def recursive_forecast(model, X_init, steps, grid_s2cnn, mask=None, lat=None, lon=None):
    B, T_in, C_in, H, W = X_init.shape
    X_seq = X_init.clone().cuda()
    forecasts = []

    for t in range(steps):
        X_s2cnn = era5_to_s2cnn_batch(X_seq, grid_s2cnn, mask)
        X_flat = X_s2cnn.reshape(B, T_in*C_in, X_s2cnn.shape[-2], X_s2cnn.shape[-1])
        with torch.no_grad():
            y_pred_s2cnn = model(X_flat)
            y_pred_era5 = s2cnn_to_era5_batch(y_pred_s2cnn, lat, lon)
        forecasts.append(y_pred_era5.cpu())

        # Update input sequence
        if T_in > 1:
            X_seq = torch.cat([X_seq[:,1:,:,:,:], y_pred_era5.unsqueeze(1)], dim=1)
        else:
            X_seq = y_pred_era5.unsqueeze(1)

    forecasts = torch.stack(forecasts, dim=1)
    return forecasts

# ------------------------------
# 7️⃣ Visualization
# ------------------------------
def plot_forecasts_global(forecasts, lat, lon, step_indices=None, variable_index=0, cmap='coolwarm'):
    if isinstance(forecasts, torch.Tensor):
        forecasts = forecasts.cpu().numpy()
    B, steps, C_out, H, W = forecasts.shape
    if step_indices is None:
        step_indices = range(steps)
    for b in range(B):
        for t in step_indices:
            data = forecasts[b,t,variable_index]
            plt.figure(figsize=(12,6))
            ax = plt.axes(projection=ccrs.PlateCarree())
            ax.coastlines()
            mesh = ax.pcolormesh(lon, lat, data, cmap=cmap, shading='auto')
            plt.title(f"Sample {b}, Forecast step {t}")
            plt.colorbar(mesh, label='Value')
            plt.show()

# ------------------------------
# 8️⃣ Metrics computation
# ------------------------------
def compute_forecast_metrics(forecasts, y_true, step_indices=None, variable_index=0):
    if isinstance(forecasts, torch.Tensor):
        forecasts = forecasts.cpu().numpy()
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().numpy()
    B, steps, C_out, H, W = forecasts.shape
    if step_indices is None:
        step_indices = range(steps)
    metrics = {}
    for t in step_indices:
        pred = forecasts[:, t, variable_index]
        true = y_true[:, t, variable_index]
        error = pred - true
        rmse = np.sqrt(np.mean(error**2))
        mae = np.mean(np.abs(error))
        mean_error = np.mean(error)
        metrics[t] = {"RMSE":rmse,"MAE":mae,"MeanError":mean_error}
    return metrics

# ------------------------------
# 9️⃣ Example: Training loop
# ------------------------------
# Example: random data for demonstration
H, W = 721, 1440
B, T_in, C_in, C_out = 2, 3, 4, 1
X_train = torch.randn(B,T_in,C_in,H,W).cuda()
Y_train = torch.randn(B,C_out,H,W).cuda()
lat = np.linspace(90,-90,H)
lon = np.linspace(0,359.75,W)
bandwidth=64
grid_s2cnn, mask_north = make_era5_to_s2cnn_grid_lat_range(lat, lon, bandwidth, 0, 90)
grid_s2cnn = grid_s2cnn.cuda()
mask_north = mask_north.cuda()

dataset = ERA5StreamingDataset(file_list=[], T_in=T_in, lat_crop=(0,90), bandwidth=bandwidth)
# Overwrite with random data
dataset.X = X_train
dataset.Y = Y_train
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

model = SphericalWeatherNet(in_channels=T_in*C_in, out_channels=C_out, bandwidth=bandwidth).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(2):
    model.train()
    for x_batch, y_batch in dataloader:
        B, T, C, Hs, Ws = x_batch.shape
        x_flat = x_batch.reshape(B,T*C,Hs,Ws).cuda()
        y_batch = y_batch.cuda()
        optimizer.zero_grad()
        y_pred_s2cnn = model(x_flat)
        y_pred_era5 = s2cnn_to_era5_batch(y_pred_s2cnn, lat, lon)
        loss = criterion(y_pred_era5, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

# ------------------------------
# 10️⃣ Recursive forecast + plot + metrics
# ------------------------------
X_init = X_train.clone()
forecast_steps = 3
forecasts = recursive_forecast(model, X_init, forecast_steps, grid_s2cnn, mask_north, lat, lon)
plot_forecasts_global(forecasts, lat, lon, step_indices=[0,1,2])
metrics = compute_forecast_metrics(forecasts, Y_train.unsqueeze(1).repeat(1,forecast_steps,1,1,1))
print(metrics)


In [1]:
import torch
import torch.nn.functional as F

class CylindricalResizer:
    def resize_cylindrical(self, x, size_hw):
        """
        Resize cylindrical images along width with circular interpolation.
        x: (..., frames, C, H, W)
        size_hw: (H1, W1)
        """
        *leading, Fm, C, H, W = x.shape
        outs = []

        # flatten leading dims
        x_flat = x.reshape(-1, Fm, C, H, W)

        for f in range(Fm):
            frame = x_flat[:, f]  # (N, C, H, W)

            # circular padding along width
            frame = F.pad(frame, (1, 1, 0, 0), mode="circular")  # (N, C, H, W+2)

            # interpolate each frame
            frame_resized = F.interpolate(
                frame, size=(size_hw[0], size_hw[1] + 2),
                mode="bicubic", align_corners=False
            )

            # remove circular padding
            frame_resized = frame_resized[:, :, :, 1:-1]  # (N, C, H1, W1)
            outs.append(frame_resized)

        # stack frames back
        output = torch.stack(outs, dim=1)  # (N, Fm, C, H1, W1)

        # reshape back to original leading dims
        output = output.reshape(*leading, Fm, C, size_hw[0], size_hw[1])
        return output


In [2]:
resizer = CylindricalResizer()

# Example 1: no leading dims
x1 = torch.randn(2, 3, 4, 8, 16)  # (B=2, frames=3, C=4, H=8, W=16)
y1 = resizer.resize_cylindrical(x1, (12, 24))
print("Example 1:", x1.shape, "->", y1.shape)

# Example 2: one leading dim
x2 = torch.randn(5, 2, 3, 4, 8, 16)  # (T=5, B=2, frames=3, C=4, H=8, W=16)
y2 = resizer.resize_cylindrical(x2, (12, 24))
print("Example 2:", x2.shape, "->", y2.shape)

# Example 3: multiple leading dims
x3 = torch.randn(2, 2, 2, 3, 4, 8, 16)  # (..., frames=3, C=4, H=8, W=16)
y3 = resizer.resize_cylindrical(x3, (10, 20))
print("Example 3:", x3.shape, "->", y3.shape)

Example 1: torch.Size([2, 3, 4, 8, 16]) -> torch.Size([2, 3, 4, 12, 24])
Example 2: torch.Size([5, 2, 3, 4, 8, 16]) -> torch.Size([5, 2, 3, 4, 12, 24])
Example 3: torch.Size([2, 2, 2, 3, 4, 8, 16]) -> torch.Size([2, 2, 2, 3, 4, 10, 20])


In [1]:
import requests
from datetime import datetime

# -------------------------------
# Yr.no (Free, no API key needed)
# -------------------------------
def get_forecast_yr(latitude, longitude):
    url = f"https://api.met.no/weatherapi/locationforecast/2.0/compact?lat={latitude}&lon={longitude}"
    headers = {
        "User-Agent": "Mike3.0-Bot/1.0"
    }
    response = requests.get(url, headers=headers)
    data = response.json()
    forecast = []
    for t in data["properties"]["timeseries"][:6]:  # next 6 hours
        dt = t["time"]
        temp = t["data"]["instant"]["details"]["air_temperature"]
        forecast.append((dt, temp))
    return forecast

In [2]:
forecast = get_forecast_yr(35, 139)

In [3]:
print(forecast)

[('2025-11-18T08:00:00Z', 10.3), ('2025-11-18T09:00:00Z', 9.1), ('2025-11-18T10:00:00Z', 8.1), ('2025-11-18T11:00:00Z', 6.8), ('2025-11-18T12:00:00Z', 6.2), ('2025-11-18T13:00:00Z', 5.7)]


In [4]:
def get_forecast_accuweather(api_key, location_key):
    url = f"http://dataservice.accuweather.com/forecasts/v1/hourly/12hour/{location_key}?apikey={api_key}&metric=true"
    response = requests.get(url)
    data = response.json()
    forecast = []
    for t in data[:6]:  # next 6 hours
        dt = t["DateTime"]
        temp = t["Temperature"]["Value"]
        forecast.append((dt, temp))
    return forecast

In [ ]:
forecast = get_forecast_yr(35, 139)